# 5.3 — Backprop, at the computer

Do the **pen-and-paper worksheet first** — this notebook exists to *vindicate* your hand-derived
gradients. The finite-difference judge (Module 3.1's nudge) will check every single one.

Run each cell with **Shift+Enter**.

In [1]:
# Setup — run this first.
import sys
sys.path.append("../../../tools")

import numpy as np

# --- built in notebooks 01 & 02 ---
def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

def forward(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1
    h  = relu(z1)
    z2 = W2 @ h + b2
    return z1, h, z2, sigmoid(z2)

def cross_entropy(y_hat, y):
    return -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))

x  = np.array([1.0, 2.0]);  y = 1.0
W1 = np.array([[0.5, -1.0], [1.0, 0.5]]); b1 = np.array([0.5, -1.0])
W2 = np.array([-1.0, 2.0]);               b2 = -2.0

## 1. The backward sweep — the lesson's seven steps, verbatim

Each line is one named step from the lesson. Compare every printed gradient with your hand
derivation: $\delta_2 = -0.5$, $\partial W_2 = (0, -0.5)$, $\partial \mathbf{h} = (0.5, -1)$,
$\delta_1 = (0, -1)$, $\partial W_1 = \begin{pmatrix} 0&0 \\ -1&-2 \end{pmatrix}$.

In [2]:
def backward(x, y, z1, h, y_hat, W2):
    d_z2 = y_hat - y                     # Steps 1-3 collapsed: prediction minus truth
    d_W2 = d_z2 * h                      # Step 4: gradient = error x (the input it multiplied)
    d_b2 = d_z2
    d_h  = W2 * d_z2                     # Step 5: error flows BACK through the weights
    d_z1 = d_h * (z1 > 0)                # Step 6: the ReLU gate — (z1 > 0) is True/False = 1/0
    d_W1 = np.outer(d_z1, x)             # Step 7: np.outer(column, row) -> the 2x2 gradient matrix
    d_b1 = d_z1
    return d_W1, d_b1, d_W2, d_b2

z1, h, z2, y_hat = forward(x, W1, b1, W2, b2)
d_W1, d_b1, d_W2, d_b2 = backward(x, y, z1, h, y_hat, W2)

print("delta_2 (y_hat - y) =", y_hat - y)
print("dL/dW2 =", d_W2, "   dL/db2 =", d_b2)
print("dL/dW1 =\n", d_W1)
print("dL/db1 =", d_b1)
# Note the zeros: everything upstream of the dead ReLU (and of h1 = 0) learned nothing.

delta_2 (y_hat - y) = -0.5
dL/dW2 = [-0.  -0.5]    dL/db2 = -0.5
dL/dW1 =
 [[ 0.  0.]
 [-1. -2.]]
dL/db1 = [ 0. -1.]


## 2. The judge: gradient checking (Module 3.1, weaponised)

A derivative is *"nudge the input, watch the output"*. So for every one of our 9 parameters:
nudge it by $0.0001$, rerun the forward pass, and see how much the loss moved. If your chain-rule
answers are right, the two columns below agree to several decimal places. **This is how real
frameworks test their backprop code.**

In [3]:
def loss_of(W1, b1, W2, b2):
    _, _, _, y_hat = forward(x, W1, b1, W2, b2)
    return cross_entropy(y_hat, y)

eps = 1e-4
params = {  # name -> (array, index, backprop's claimed gradient)
    "W1[0,0]": (W1, (0, 0), d_W1[0, 0]), "W1[0,1]": (W1, (0, 1), d_W1[0, 1]),
    "W1[1,0]": (W1, (1, 0), d_W1[1, 0]), "W1[1,1]": (W1, (1, 1), d_W1[1, 1]),
    "b1[0]":   (b1, (0,),   d_b1[0]),    "b1[1]":   (b1, (1,),   d_b1[1]),
    "W2[0]":   (W2, (0,),   d_W2[0]),    "W2[1]":   (W2, (1,),   d_W2[1]),
}

print(f"{'param':>8} | {'backprop':>9} | {'nudge test':>10}")
print("-" * 35)
base = loss_of(W1, b1, W2, b2)
for name, (arr, idx, claimed) in params.items():
    arr[idx] += eps                      # nudge...
    numeric = (loss_of(W1, b1, W2, b2) - base) / eps
    arr[idx] -= eps                      # ...and undo the nudge
    print(f"{name:>8} | {claimed:9.4f} | {numeric:10.4f}")
print("\nEvery row agrees: the chain rule you did by hand is correct.")

   param |  backprop | nudge test
-----------------------------------
 W1[0,0] |    0.0000 |     0.0000
 W1[0,1] |    0.0000 |     0.0000
 W1[1,0] |   -1.0000 |    -0.9999
 W1[1,1] |   -2.0000 |    -1.9998
   b1[0] |    0.0000 |     0.0000
   b1[1] |   -1.0000 |    -1.0000
   W2[0] |   -0.0000 |     0.0000
   W2[1] |   -0.5000 |    -0.5000

Every row agrees: the chain rule you did by hand is correct.


In [4]:
# YOUR TURN — be the judge yourself.
# 1) b2 is missing from the table. Your hand answer says dL/db2 = -0.5.
#    Nudge b2 by 0.0001, recompute the loss, and confirm (mind: b2 is a plain float here,
#    so just call loss_of(W1, b1, W2, b2 + 0.0001)).

# 2) Worksheet problem 15 asked you to PREDICT a new loss after a +0.01 nudge.
#    Test your prediction on this network: pick any weight, predict loss_new using
#    loss_new ~ loss_old + gradient * 0.01, then measure it for real.

## 3. Watch a dead neuron wake up

For $\mathbf{x} = (1, 2)$, neuron 1 was off and its entire row of $W_1$ got zero gradient.
Worksheet problem 13 asked which inputs wake it. Feed one in — and watch the gradient come alive.

In [5]:
x_wake = np.array([2.0, 0.0])          # 0.5(2) - 0 + 0.5 = 1.5 > 0 -> neuron 1 fires

z1w, hw, _, y_hatw = forward(x_wake, W1, b1, W2, b2)
gW1, gb1, gW2, gb2 = backward(x_wake, y, z1w, hw, y_hatw, W2)

print("z1 =", z1w, " -> h =", hw)
print("dL/dW1 =\n", gW1)
# Row 1 is nonzero now: THIS example can teach neuron 1 something.
# Different inputs light up different paths — that's why training needs varied data,
# and why a neuron that's off for EVERY input is dead forever (worksheet problem 16).

z1 = [1.5 1. ]  -> h = [1.5 1. ]
dL/dW1 =
 [[ 1.63514895  0.        ]
 [-3.2702979  -0.        ]]


Backprop: not magic, just Module 3.3 with good bookkeeping — and now numerically verified,
gradient by gradient, by your own judge.

---
*Done? Photograph your worksheet into `scans/inbox/` and tell Claude.
Next: 5.4 — the training loop. Bring popcorn; your math is about to learn.*